# 12 — วิเคราะห์พายุที่ส่งผลต่อประเทศไทย ด้วยข้อมูล ERA5

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**บทเสริม (extension) · ใช้เวลาประมาณ 3–4 ชั่วโมง**

---

## เราจะทำอะไรกัน

เลือกเหตุการณ์ที่กระทบประเทศไทยมา **4 เหตุการณ์จริง** แล้ววิเคราะห์ลักษณะอากาศ
ตั้งแต่ **5 วันก่อนเกิดเหตุ จนถึง 2 วันหลัง** (D-5 ถึง D+2)
เพื่อตอบคำถามว่า — ก่อนที่ข่าวจะประกาศเตือน บรรยากาศส่งสัญญาณอะไรไว้บ้าง

| | เหตุการณ์ | D0 (วันเกิดเหตุ) | ลักษณะเด่น |
|---|---|---|---|
| 1 | พายุโซนร้อน **เตี้ยนหมู่** (Dianmu) | 24 ก.ย. 2564 | อ่อนกำลังเป็นดีเปรสชัน เคลื่อนเข้าปกคลุมอีสานตอนล่าง |
| 2 | ไต้ฝุ่น/โซนร้อน **โนรู** (Noru) | 28 ก.ย. 2565 | ขึ้นฝั่งเวียดนามในสถานะไต้ฝุ่น แล้วเข้าไทยที่โขงเจียม |
| 3 | ไต้ฝุ่น **ยางิ** (Yagi) | 7 ก.ย. 2567 | **ศูนย์กลางไม่เข้าไทยเลย** แต่ขอบพายุทำให้ภาคเหนือตอนบนฝนหนัก |
| 4 | **น้ำท่วมกรุงเทพมหานคร** | 25 ก.ย. 2569 | **ไม่มีพายุที่มีชื่อเลย** เป็นหย่อมความกดอากาศต่ำบนร่องมรสุม |

เรียงสี่เหตุการณ์แบบนี้เพื่อไล่ประเด็นทีละขั้น

- เคส 1–2: พายุเข้าตรง ๆ ต่างกันที่ความแรง
- เคส 3: **พายุไม่เข้า ไม่ได้แปลว่าปลอดภัย**
- เคส 4: **ภัยพิบัติหนักที่สุดอาจไม่มีพายุที่มีชื่อเรียกเลยด้วยซ้ำ**

เคสที่ 4 ยังต่างจากสามเคสแรกตรงที่เป็น **ความเสียหายในเมือง** ซึ่งปัจจัยไม่ได้มีแค่ปริมาณฝน
แต่รวมถึงความสามารถในการระบายน้ำของเมืองด้วย — เป็นโจทย์ภูมิศาสตร์เมืองโดยตรง

---

### สิ่งที่จะได้จากบทนี้

- เปิดไฟล์ **NetCDF** ด้วย `xarray` เลือกเวลา ระดับความกดอากาศ และตัดโดเมน
- วาดแผนที่อากาศด้วย `cartopy` — แรเงา เส้นชั้น wind barb streamline
- **คำนวณเส้นทางพายุจากข้อมูลเอง** แล้วเทียบกับที่ข่าวรายงาน
- อ่านลำดับเหตุการณ์จากแผนที่: ก่อนเกิด → ระหว่างเกิด → หลังเกิด
- เปรียบเทียบสี่เหตุการณ์อย่างมีหลักฐานเชิงปริมาณ
- เทียบข้อมูล reanalysis กับค่าที่สถานีตรวจวัดรายงาน และอธิบายว่าทำไมถึงต่างกัน

**ต้องผ่านบทที่ 09 (pandas) และ 10 (matplotlib) มาก่อน**

---

### แหล่งอ้างอิง

ลำดับเหตุการณ์ทั้งหมดอ้างอิงจากประกาศกรมอุตุนิยมวิทยาและรายงานข่าว
ดูรายการเต็มของแต่ละเคสได้จาก `ec.describe_case()` ในหัวข้อกรณีศึกษา

ข้อมูลอุตุนิยมวิทยาเป็น **ERA5 reanalysis ของ ECMWF** ซึ่งเป็นข้อมูลจริง
ถูกตัดเฉพาะโดเมนและช่วงเวลาที่ใช้ แล้วลดความละเอียดเหลือ 0.5 องศา เพื่อให้ไฟล์เล็กพอ

> เมื่อนำผลไปใช้ในรายงานหรืองานวิชาการ **ต้องอ้างอิงแหล่งข้อมูล**
> Hersbach, H. et al. (2020). The ERA5 global reanalysis.
> *Quarterly Journal of the Royal Meteorological Society*, 146(730), 1999–2049.

## 0. เตรียมเครื่องมือและข้อมูล

Colab มี `xarray` กับ `netCDF4` มาให้แล้ว แต่ไม่มี `cartopy` ต้องติดตั้งเอง
ใช้เวลาประมาณ 1 นาที รันครั้งเดียวต่อ session

In [ ]:
!pip install -q cartopy
print("ติดตั้ง cartopy เรียบร้อย")

In [ ]:
# ===== ดาวน์โหลด module ช่วยงานและข้อมูลกรณีศึกษา (รันครั้งเดียวต่อ session) =====
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, urllib.request

for name in ["era5_utils.py", "era5_cases.py"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + "utils/" + name, name)

import era5_cases as ec

os.makedirs("data/cases", exist_ok=True)
CASES_READY = True
for case_id in ec.CASE_ORDER:
    target = f"data/cases/{case_id}.nc"
    if not os.path.exists(target):
        try:
            urllib.request.urlretrieve(BASE + f"data/cases/{case_id}.nc", target)
            print("ดาวน์โหลดแล้ว:", case_id)
        except Exception as exc:
            CASES_READY = False
            print("ยังไม่มีข้อมูลของเคส", case_id, "->", exc)

if CASES_READY:
    print(f"\nข้อมูลกรณีศึกษาพร้อมใช้งานครบทั้ง {len(ec.CASE_ORDER)} เหตุการณ์")
else:
    print("\nยังดาวน์โหลดข้อมูลไม่ครบ")
    print("ผู้สอนต้องรัน utils/fetch_case_data.py หนึ่งครั้ง")
    print("แล้ว commit โฟลเดอร์ data/cases/ ขึ้น GitHub ก่อน (ดูภาคผนวก ก ท้ายบท)")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import era5_utils as eu
import era5_cases as ec

eu.setup_thai_font()
plt.rcParams["figure.dpi"] = 110
print("พร้อมทำงาน")

## 1. gridded data ต่างจากตารางอย่างไร

ข้อมูลฝนในบทที่ 09 เป็น **ตาราง** หนึ่งแถวคือหนึ่งการตรวจวัดที่หนึ่งจุด หนึ่งเวลา
เหมาะกับข้อมูลสถานีซึ่งกระจายไม่สม่ำเสมอ

ERA5 เป็น **กริด** ค่าถูกวางบนตารางสี่เหลี่ยมที่ครอบคลุมทุกจุด
ข้อมูลหนึ่งตัวแปรจึงมี 4 มิติ: `time × level × latitude × longitude`

| | pandas DataFrame | xarray Dataset |
|---|---|---|
| รูปร่าง | 2 มิติ (แถว × คอลัมน์) | กี่มิติก็ได้ |
| เลือกข้อมูลด้วย | ตำแหน่งแถว หรือเงื่อนไข | **ชื่อมิติและค่าพิกัดจริง** |
| เหมาะกับ | ข้อมูลสถานี ข้อมูลสำรวจ | reanalysis, ภาพดาวเทียม, ผลลัพธ์แบบจำลอง |

> ถ้าเคยใช้ ArcGIS/QGIS มาก่อน: `xarray.Dataset` คือ **multiband raster ที่มีแกนเวลาและแกนความสูงเพิ่มเข้ามา**

In [ ]:
ds1 = ec.load_case("dianmu2021")
ds1

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

กดสามเหลี่ยมหน้าชื่อตัวแปรเพื่อดูรายละเอียด สังเกต 3 ส่วน — **Dimensions**, **Coordinates**, **Data variables**

สังเกตว่ามี coordinate พิเศษชื่อ `day_offset` ซึ่ง `load_case()` เพิ่มให้
เป็นจำนวนวันเทียบกับวันเกิดเหตุ: `-5` คือห้าวันก่อน `0` คือวันเกิดเหตุ `+2` คือสองวันหลัง

In [ ]:
print("มิติ      :", dict(ds1.sizes))
print("ตัวแปร    :", list(ds1.data_vars))
print("ระดับความกดอากาศ:", ds1["level"].values, "hPa")
print("day_offset:", sorted(set(ds1["day_offset"].values)))
print("โดเมน     : lon", float(ds1.longitude.min()), "ถึง", float(ds1.longitude.max()),
      "| lat", float(ds1.latitude.min()), "ถึง", float(ds1.latitude.max()))
print()
print("ที่มาของข้อมูล:", ds1.attrs.get("source"))
print("การประมวลผล :", ds1.attrs.get("processing"))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ตัวแปรที่มีในไฟล์

ชื่อย่อเหล่านี้เป็นชื่อมาตรฐานของ ECMWF ควรจำไว้ เพราะเจอทุกครั้งที่ใช้ ERA5

| ชื่อ | ความหมาย | หน่วยในไฟล์ | หน่วยที่คนอ่าน |
|---|---|---|---|
| `t2m` | อุณหภูมิที่ความสูง 2 เมตร | K | °C |
| `msl` | ความกดอากาศที่ระดับน้ำทะเล | Pa | hPa |
| `tp` | **ฝนสะสมในช่วง 6 ชั่วโมงที่สิ้นสุด ณ เวลานั้น** | mm | mm |
| `u` | ลมในแนวตะวันออก–ตะวันตก | m s⁻¹ | m s⁻¹ |
| `v` | ลมในแนวเหนือ–ใต้ | m s⁻¹ | m s⁻¹ |
| `z` | geopotential | m² s⁻² | gpm (หาร 9.80665) |
| `r` | ความชื้นสัมพัทธ์ | % | % |

⚠️ **สองเรื่องที่พลาดกันบ่อยที่สุด**

1. **`u` เป็นบวก = ลมพัดไปทางตะวันออก** (westerly) ไม่ใช่พัดมาจากตะวันออก
2. `tp` ต้นฉบับของ ERA5 เป็นฝนสะสม **รายชั่วโมง หน่วยเมตร** ถ้าสุ่มเอาทุก 6 ชั่วโมงตรง ๆ
   ฝนจะหายไป 5 ใน 6 ส่วน — ไฟล์ของเรารวมมาให้แล้วและแปลงเป็นมิลลิเมตร

## 2. เลือกข้อมูล และแปลงหน่วย

- `.sel()` เลือกด้วย **ค่าจริง** เช่น `level=850`, `time="2021-09-24T12:00"`
- `.isel()` เลือกด้วย **ลำดับ** เช่น `time=0`
- `method="nearest"` ใช้เมื่อค่าที่ขอไม่ตรงกริดพอดี เช่นพิกัดจังหวัด

In [ ]:
# เลือกเวลาเดียว -> เหลือ 2 มิติ พร้อมพล็อตเป็นแผนที่
t_one = ds1["t2m"].sel(time="2021-09-24T12:00")
print("หลังเลือกเวลา:", t_one.dims, t_one.shape)

# เลือกทั้งเวลาและระดับ
u850 = ds1["u"].sel(time="2021-09-24T12:00", level=850)
print("ลม 850 hPa  :", u850.dims, u850.shape)

# เลือกทั้งวัน (ได้ 4 ช่วงเวลา)
one_day = ds1.sel(time="2021-09-24")
print("ทั้งวัน      :", dict(one_day.sizes))

# ค่าที่จุดเดียว — พิกัดจังหวัดอุบลราชธานี
point = ds1["tp"].sel(latitude=15.25, longitude=104.87, method="nearest")
print("\nฝนราย 6 ชม. ที่จุดใกล้อุบลราชธานี 8 ค่าแรก (mm):")
print(point.values[:8].round(2))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
# แปลงหน่วยก่อนพล็อตเสมอ ไม่มีใครอ่านแผนที่หน่วยเคลวินหรือปาสกาล
snap = ds1.sel(time="2021-09-24T12:00")

t_c = eu.to_celsius(snap["t2m"])
p_hpa = eu.to_hpa(snap["msl"])
z500 = eu.to_gpm(snap["z"].sel(level=500))

print(f"อุณหภูมิ      {float(t_c.min()):6.1f} ถึง {float(t_c.max()):6.1f} °C")
print(f"ความกดอากาศ  {float(p_hpa.min()):6.1f} ถึง {float(p_hpa.max()):6.1f} hPa")
print(f"ความสูง 500  {float(z500.min()):6.0f} ถึง {float(z500.max()):6.0f} gpm")

# ค่าเฉลี่ยเชิงพื้นที่ต้องถ่วงน้ำหนักด้วย cos(latitude)
# เพราะกริดใกล้ขั้วโลกมีพื้นที่ต่อเซลล์เล็กกว่ากริดใกล้ศูนย์สูตร
print(f"\nอุณหภูมิเฉลี่ยทั้งโดเมน ถ่วงน้ำหนัก = {eu.area_mean(t_c):.2f} °C")
print(f"ถ้าเฉลี่ยตรง ๆ ไม่ถ่วงน้ำหนัก       = {float(t_c.mean()):.2f} °C")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3. แผนที่ใบแรก

ขั้นตอนของแผนที่หนึ่งใบมี 4 ส่วนเสมอ

1. สร้างแกนที่มี **projection** — `ccrs.PlateCarree()` คือพิกัดภูมิศาสตร์ธรรมดา
2. กำหนดขอบเขต ใส่เส้นชายฝั่ง เส้นพรมแดน gridline
3. วาดข้อมูลทับ โดย **ต้องระบุ `transform=ccrs.PlateCarree()`** เสมอ
4. ใส่ชื่อแผนที่ colorbar แล้วบันทึก

> `transform=` บอก cartopy ว่า **ข้อมูลอยู่ในระบบพิกัดอะไร** ส่วน projection ของแกนคือ
> **ระบบที่จะวาดออกมา** ถ้าลืม cartopy จะเดาว่าข้อมูลอยู่ในระบบเดียวกับแกน
> พอเปลี่ยน projection เมื่อไรแผนที่จะเพี้ยนทันที

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([95, 115, 5, 25], crs=ccrs.PlateCarree())

# cartopy ไม่ได้ติดตั้งแผนที่ฐานมาด้วย แต่ไปโหลดจาก Natural Earth ตอนวาดรูป
# ถ้าเครือข่ายบล็อก จะพังตอน plt.show() ซึ่งดักยาก จึงบังคับโหลดตรงนี้ก่อน
try:
    for feat in (cfeature.COASTLINE, cfeature.BORDERS, cfeature.OCEAN, cfeature.LAND):
        next(iter(feat.geometries()), None)
    ax.add_feature(cfeature.OCEAN, facecolor="#eaf3fb")
    ax.add_feature(cfeature.LAND, facecolor="#f7f5f0")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
except Exception as exc:
    print("โหลดแผนที่ฐานไม่ได้ ->", exc)
    print("แผนที่จะไม่มีเส้นชายฝั่ง แต่ข้อมูลที่พล็อตยังถูกต้องทุกอย่าง")

gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                  alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False

cs = ax.contour(p_hpa["longitude"], p_hpa["latitude"], p_hpa.values,
                levels=np.arange(960, 1021, 4), colors="black",
                linewidths=1.0, transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=8, fmt="%d")

ax.set_title("เตี้ยนหมู่ — ความกดอากาศที่ระดับน้ำทะเล (hPa)\n2021-09-24 12 UTC",
             loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ลมสัมพันธ์กับความกดอากาศอย่างไร

`barbs()` วาดสัญลักษณ์ลมแบบที่ใช้ในแผนที่อากาศจริง — ก้านชี้ไป **ทิศที่ลมพัดมาจาก**
ขีดสั้น = 5 นอต · ขีดยาว = 10 นอต · ธงสามเหลี่ยม = 50 นอต
**ต้องแปลงจาก m/s เป็นนอตก่อน** (× 1.94384) ซึ่ง `eu.barbs()` ทำให้แล้ว

In [ ]:
d850 = ds1.sel(time="2021-09-24T12:00", level=850)
ws850 = eu.wind_speed(d850["u"], d850["v"])

ax = eu.make_map_axes(domain=(95, 115, 5, 25), figsize=(9.5, 7.5))
cf = eu.shade(ax, ws850, cmap="YlGnBu", levels=np.arange(0, 31, 2.5))
eu.contour(ax, p_hpa, levels=np.arange(960, 1021, 4), color="black", linewidth=0.8)
eu.barbs(ax, d850["u"], d850["v"], step=3, color="black")

eu.finish(ax, title="เตี้ยนหมู่ — ลม 850 hPa และความกดอากาศผิวพื้น\n2021-09-24 12 UTC",
          mappable=cf, cbar_label="ความเร็วลม (m s$^{-1}$)")
plt.show()

print(f"ลมแรงสุดในโดเมน {float(ws850.max()):.1f} m/s = {float(ws850.max())*1.94384:.0f} นอต")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

สิ่งที่ควรเห็น: **ลมพัดวนทวนเข็มนาฬิการอบศูนย์กลางความกดอากาศต่ำ**

นี่คือผลของ **Coriolis force** ในซีกโลกเหนือ อากาศไม่ได้ไหลตรงจากความกดอากาศสูงไปต่ำ
เหมือนน้ำไหลลงที่ต่ำ แต่ถูกเบี่ยงไปทางขวาจนไหลเกือบขนานกับ isobar (**geostrophic balance**)

สองข้อที่ใช้อ่านแผนที่อากาศได้ทันที
- isobar ชิดกัน = pressure gradient แรง = **ลมแรง**
- ซีกโลกใต้หมุนกลับทาง เพราะ Coriolis เบี่ยงไปทางซ้าย

---

# กรณีศึกษาที่ 1 — พายุโซนร้อนเตี้ยนหมู่ (24 ก.ย. 2564)

## 1.1 ลำดับเหตุการณ์จากข่าว

อ่านข่าวก่อนดูข้อมูลเสมอ เพื่อให้รู้ว่ากำลังมองหาอะไร

In [ ]:
print(ec.describe_case("dianmu2021"))

## 1.2 เส้นทางพายุ — คำนวณจากข้อมูลเอง

เราจะไม่ลอกเส้นทางจากข่าว แต่ให้โปรแกรม **หาตำแหน่งความกดอากาศต่ำสุด** ในแต่ละช่วงเวลา
แล้วต่อเป็นเส้นทาง จากนั้นค่อยเทียบกับที่ข่าวรายงานว่าตรงกันไหม

นี่คือวิธีที่ใช้จริงในการวิเคราะห์ย้อนหลัง และเป็นการตรวจสอบข้อมูลไปในตัว

In [ ]:
ax, track1 = ec.plot_track(ds1, "dianmu2021")
plt.show()

print("เส้นทางที่คำนวณได้ (แสดงเฉพาะเวลา 00 UTC ของแต่ละวัน)")
print(track1[track1["time"].dt.hour == 0].to_string(index=False))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

**ตรวจสอบกับข่าว:** ข่าวระบุว่าวันที่ 24 ก.ย. 2564 ศูนย์กลางเคลื่อนเข้าปกคลุม
มุกดาหาร อำนาจเจริญ และอุบลราชธานี ซึ่งอยู่ประมาณ **15–16.5°N, 104.5–105.5°E**

ดูในตารางว่าตำแหน่งที่คำนวณได้ในวัน D0 ใกล้เคียงหรือไม่

> ถ้าไม่ตรงนัก เป็นเรื่องปกติ — วิธีนี้หาเพียง "จุดต่ำสุดของความกดอากาศในโดเมน"
> ถ้าในโดเมนมีหย่อมความกดอากาศต่ำอื่นที่ลึกกว่า มันจะจับผิดตัว
> วิธีแก้คือจำกัดขอบเขตการค้นหาให้แคบลงด้วย `search_domain`
> **การที่เครื่องมือมีข้อจำกัด ไม่ได้แปลว่าใช้ไม่ได้ แต่แปลว่าต้องรู้ว่ามันพลาดตรงไหน**

## 1.3 ความรุนแรงตามเวลา

กราฟนี้ตอบคำถามว่า พายุ **ทวีกำลังตอนไหน และอ่อนกำลังตอนไหน**

In [ ]:
ec.plot_intensity(ds1, "dianmu2021")
plt.show()

print(ec.case_summary(ds1, "dianmu2021").to_string(index=False))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

**สิ่งที่ควรสังเกต**

- ความกดอากาศที่ศูนย์กลาง **ลดลง** = พายุทวีกำลัง · **เพิ่มขึ้น** = อ่อนกำลัง
  (กราฟกลับแกนไว้แล้ว เส้นลงต่ำ = พายุแรงขึ้น)
- พายุมักอ่อนกำลังเร็วหลังขึ้นฝั่ง เพราะขาดความชื้นและความร้อนจากผิวทะเล
  และมีแรงเสียดทานจากพื้นดิน
- แต่ให้ดูคอลัมน์ `rain_max_mm` ด้วย — **ฝนหนักที่สุดมักไม่ได้เกิดวันเดียวกับที่ลมแรงที่สุด**

## 1.4 ภาพรวมรายวัน D-5 ถึง D+2

แผนที่ 8 ใบเรียงกันคือหัวใจของบทนี้ ทำให้เห็นลำดับ **ก่อน → ระหว่าง → หลัง** ในภาพเดียว
ทุกใบใช้ระดับสีและเส้นชั้นชุดเดียวกัน ไม่งั้นเปรียบเทียบกันไม่ได้

In [ ]:
ec.panel_daily(ds1, "dianmu2021", variable="rain", savepath="dianmu_rain.png")
plt.show()

In [ ]:
ec.panel_daily(ds1, "dianmu2021", variable="wind")
plt.show()

In [ ]:
ec.panel_daily(ds1, "dianmu2021", variable="rh")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 1.5 ค่าที่จุดจังหวัดที่ได้รับผลกระทบ

แผนที่บอกภาพรวม แต่คำถามของนักภูมิศาสตร์มักเป็น "ที่จังหวัดเรา เกิดอะไรขึ้น"
ตรงนี้คือสะพานเชื่อมกลับไปหา pandas ที่เรียนมาแล้วในบทที่ 09

In [ ]:
pts1 = ec.point_series(ds1, ec.case_info("dianmu2021")["focus_points"])
display(pts1.head())

# ฝนรายวันของแต่ละจังหวัด
daily_pt = (pts1.assign(date=pts1["time"].dt.date)
                .groupby(["place", "date"])["rain_mm"].sum()
                .unstack(0).round(1))
display(daily_pt)

fig, ax = plt.subplots(figsize=(11, 4.5))
daily_pt.plot(kind="bar", ax=ax)
ax.axvline(list(daily_pt.index).index(pd.Timestamp("2021-09-24").date()),
           color="crimson", linestyle=":", linewidth=2)
ax.set_title("เตี้ยนหมู่ — ฝนรายวันที่จุดจังหวัดต่าง ๆ (เส้นแดง = D0)", loc="left")
ax.set_xlabel("วันที่")
ax.set_ylabel("ฝนรายวัน (mm)")
ax.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 1.6 อธิบายผลลัพธ์

> **หมายเหตุสำคัญ:** หัวข้อนี้ไม่ได้เขียนข้อสรุปสำเร็จรูปไว้ให้ เพราะข้อสรุปต้องมาจาก
> ตัวเลขที่รันออกมาจริง ไม่ใช่จากสิ่งที่คาดว่าน่าจะเป็น
> ให้ใช้กรอบคำถามข้างล่างเป็นตัวนำ แล้วเขียนข้อสรุปด้วยตัวเอง

**คำถามนำ**

1. **ก่อนเกิดเหตุ (D-5 ถึง D-1)** — ความกดอากาศที่ศูนย์กลางเปลี่ยนแปลงอย่างไร
   ความชื้นที่ 850 hPa เหนือภาคอีสานเริ่มเพิ่มขึ้นตั้งแต่วันไหน
   มีสัญญาณอะไรที่เห็นได้ **ก่อน** กรมอุตุฯ ออกประกาศฉบับแรกหรือไม่
2. **วันเกิดเหตุ (D0)** — ตำแหน่งศูนย์กลางที่คำนวณได้ตรงกับที่ข่าวรายงานไหม
   ฝนหนักที่สุดอยู่ตรงศูนย์กลางพอดี หรือเยื้องไปด้านใดด้านหนึ่ง
3. **หลังเกิดเหตุ (D+1, D+2)** — ความกดอากาศกลับสู่ปกติเร็วแค่ไหน
   ฝนหยุดพร้อมกับที่พายุอ่อนกำลัง หรือยังตกต่อ

**ประเด็นที่ควรเขียนถึงให้ได้**

- ความสัมพันธ์ระหว่าง **ความแรงของลม** กับ **ปริมาณฝน** — สองอย่างนี้ไม่ได้สูงสุดพร้อมกันเสมอไป
- ทิศทางลมที่ 850 hPa บอกอะไรเกี่ยวกับ **แหล่งความชื้น** ที่ป้อนเข้าสู่ระบบ
- เหตุใดพายุที่ "อ่อนกำลังเป็นดีเปรสชันแล้ว" จึงยังทำให้เกิดน้ำท่วมใหญ่ได้

### ✍️ ข้อสรุปของฉัน — กรณีเตี้ยนหมู่

(เขียนที่นี่ 5–10 บรรทัด อ้างตัวเลขจากตารางและแผนที่ข้างบนประกอบด้วย)

---

---

# กรณีศึกษาที่ 2 — ไต้ฝุ่น/โซนร้อนโนรู (28 ก.ย. 2565)

คราวนี้ใช้ฟังก์ชันชุดเดิมทั้งหมด เปลี่ยนแค่ `case_id`
นี่คือข้อดีของการเขียนโค้ดเป็นฟังก์ชันตั้งแต่บทที่ 07 — **เขียนครั้งเดียว ใช้ได้กับทุกเหตุการณ์**

โนรูเคยมีสถานะเป็น **ไต้ฝุ่น** ก่อนขึ้นฝั่งเวียดนาม ต่างจากเตี้ยนหมู่ที่เป็นพายุโซนร้อน
ลองดูว่าความแรงที่ต่างกันส่งผลต่อลักษณะอากาศในไทยอย่างไร

In [ ]:
print(ec.describe_case("noru2022"))

In [ ]:
ds2 = ec.load_case("noru2022")
ax, track2 = ec.plot_track(ds2, "noru2022")
plt.show()
print(track2[track2["time"].dt.hour == 0].to_string(index=False))

In [ ]:
ec.plot_intensity(ds2, "noru2022")
plt.show()
print(ec.case_summary(ds2, "noru2022").to_string(index=False))

In [ ]:
ec.panel_daily(ds2, "noru2022", variable="rain")
plt.show()

In [ ]:
ec.panel_daily(ds2, "noru2022", variable="rh")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 2.6 อธิบายผลลัพธ์

**คำถามนำ**

1. เทียบความกดอากาศต่ำสุดของโนรูกับเตี้ยนหมู่ — ต่างกันกี่ hPa
   แล้วฝนสูงสุดในพื้นที่ศึกษาต่างกันตามสัดส่วนนั้นหรือไม่
2. ข่าวระบุว่าโนรูขึ้นฝั่งดานังเช้าวันที่ 28 แล้วถึงโขงเจียมเย็นวันเดียวกัน
   จากเส้นทางที่คำนวณได้ **พายุเคลื่อนที่เร็วเท่าไร (km/h)** และความเร็วนี้
   มีผลต่อปริมาณฝนสะสมอย่างไร (ใบ้: เคลื่อนช้า = ฝนตกที่เดิมนานกว่า)
3. พายุที่แรงกว่าตอนอยู่กลางทะเล **ไม่จำเป็นต้องทำให้ฝนในไทยมากกว่า** — จากข้อมูลที่เห็น
   อะไรน่าจะเป็นตัวกำหนดปริมาณฝนในไทยมากกว่าความแรงสูงสุดของพายุ

### ✍️ ข้อสรุปของฉัน — กรณีโนรู

(เขียนที่นี่)

---

---

# กรณีศึกษาที่ 3 — ไต้ฝุ่นยางิ (7 ก.ย. 2567)

เคสนี้ต่างจากสองเคสแรกอย่างสำคัญ: **ศูนย์กลางพายุไม่เคยเข้าประเทศไทยเลย**
ขึ้นฝั่งเวียดนามตอนบนแล้วสลายตัวใน สปป.ลาว

แต่ภาคเหนือตอนบนและอีสานตอนบนยังมีฝนตกหนักถึงหนักมาก — **ทำไม**

นี่คือบทเรียนที่สำคัญที่สุดของทั้งบท คนทั่วไปมักเข้าใจว่า "พายุไม่เข้า = ปลอดภัย"
ซึ่งเป็นความเข้าใจผิดที่นำไปสู่การประเมินความเสี่ยงต่ำกว่าความเป็นจริง

In [ ]:
print(ec.describe_case("yagi2024"))

In [ ]:
ds3 = ec.load_case("yagi2024")
ax, track3 = ec.plot_track(ds3, "yagi2024")
plt.show()
print(track3[track3["time"].dt.hour == 0].to_string(index=False))

In [ ]:
ec.panel_daily(ds3, "yagi2024", variable="rain")
plt.show()

### กุญแจของเคสนี้อยู่ที่สนามลม ไม่ใช่ตำแหน่งศูนย์กลาง

พายุที่หมุนทวนเข็มนาฬิกาจะ **ดึงลมตะวันตกเฉียงใต้ที่ชื้นจากทะเลอันดามันและอ่าวไทย**
ให้พัดแรงขึ้นและพุ่งเข้าหาภาคเหนือ แม้ตัวพายุจะอยู่ไกลออกไปหลายร้อยกิโลเมตร

เมื่อลมชื้นปะทะเทือกเขาทางภาคเหนือ อากาศถูกยกตัว (**orographic lift**) จึงเกิดฝนหนัก

In [ ]:
day0 = ec.case_info("yagi2024")["d0"]
d = ds3.sel(time=f"{day0}T06:00", level=850)

ax = eu.make_map_axes(domain=ec.case_info("yagi2024")["domain"], figsize=(10, 8))
if ec.has_rh(ds3):
    cf = eu.shade(ax, d["r"], cmap="BrBG", levels=np.arange(20, 101, 5))
else:
    print("ไฟล์นี้ไม่มีความชื้นสัมพัทธ์ จึงแรเงาด้วยความเร็วลมแทน")
    cf = eu.shade(ax, eu.wind_speed(d["u"], d["v"]), cmap="YlGnBu",
                  levels=np.arange(0, 31, 2.5))
eu.streamlines(ax, d["u"], d["v"], density=1.8)
eu.contour(ax, eu.to_hpa(d["msl"]), levels=np.arange(960, 1021, 4),
           color="black", linewidth=0.7)

for name, (lat, lon) in ec.case_info("yagi2024")["focus_points"].items():
    ax.plot(lon, lat, "r^", markersize=9, transform=ccrs.PlateCarree(), zorder=6)
    ax.text(lon + 0.2, lat - 0.5, name, fontsize=9, color="darkred",
            transform=ccrs.PlateCarree(), zorder=6)

eu.finish(ax, title=f"ยางิ — สนามความชื้น/ลม และเส้นกระแสลม 850 hPa\n{day0} 06 UTC",
          mappable=cf,
          cbar_label="ความชื้นสัมพัทธ์ (%)" if ec.has_rh(ds3)
                     else "ความเร็วลม (m s$^{-1}$)")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

In [ ]:
# ลมและฝนเหนือภาคเหนือตอนบน ก่อน ระหว่าง และหลังเหตุการณ์
north = eu.subset(ds3, domain=ec.case_info("yagi2024")["focus_domain"])
ws_north = eu.wind_speed(north["u"].sel(level=850), north["v"].sel(level=850))

series = pd.DataFrame({
    "time": pd.to_datetime(ds3["time"].values),
    "day_offset": ds3["day_offset"].values,
    "ws850_mean": [round(float(eu.area_mean(ws_north.sel(time=t))), 2)
                   for t in ds3["time"].values],
    "rh850_mean": ([round(float(eu.area_mean(north["r"].sel(level=850, time=t))), 1)
                    for t in ds3["time"].values] if ec.has_rh(ds3) else np.nan),
    "rain_mean": [round(float(north["tp"].sel(time=t).mean()), 2)
                  for t in ds3["time"].values],
})
display(series)

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(series["time"], series["ws850_mean"], "o-", color="navy")
ax1.set_ylabel("ความเร็วลมเฉลี่ย 850 hPa (m s$^{-1}$)", color="navy")
ax1.tick_params(axis="y", labelcolor="navy")

ax2 = ax1.twinx()
ax2.bar(series["time"], series["rain_mean"], width=0.2, alpha=0.4, color="teal")
ax2.set_ylabel("ฝนเฉลี่ยราย 6 ชม. (mm)", color="teal")
ax2.tick_params(axis="y", labelcolor="teal")

ax1.axvline(pd.Timestamp(day0), color="crimson", linestyle=":", linewidth=2)
ax1.set_title("ยางิ — ลมและฝนเหนือภาคเหนือตอนบน (เส้นแดง = D0)", loc="left")
ax1.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3.5 อธิบายผลลัพธ์

**คำถามนำ**

1. ศูนย์กลางพายุอยู่ห่างจากเชียงรายกี่กิโลเมตรในวัน D0
   (ใช้ `eu.haversine` ที่เขียนไว้ตั้งแต่บทที่ 07 คำนวณจาก `track3`)
2. ความเร็วลม 850 hPa เหนือภาคเหนือตอนบน **เริ่มเพิ่มขึ้นตั้งแต่วันไหน**
   เทียบกับวันที่ฝนเริ่มตกหนัก — อย่างไหนมาก่อน และห่างกันกี่ชั่วโมง
3. ทิศลมที่ 850 hPa เหนือภาคเหนือเป็นทิศใด ลมทิศนั้นพัดผ่านพื้นที่ใดมาก่อน
   จึงพาความชื้นมาได้มาก
4. จากทั้งสามเคส ถ้าใช้เกณฑ์เตือนภัยว่า "ศูนย์กลางพายุจะเข้าจังหวัดไหน"
   เพียงอย่างเดียว จะพลาดอะไรไปบ้าง ควรเพิ่มตัวแปรใดเข้าไปในเกณฑ์

### ✍️ ข้อสรุปของฉัน — กรณียางิ

(เขียนที่นี่)

---

---

# กรณีศึกษาที่ 4 — น้ำท่วมกรุงเทพมหานคร (25 ก.ย. 2569)

สามเคสแรกมีพายุที่มีชื่อเรียกทั้งหมด เคสนี้ **ไม่มี**

สาเหตุคือ **หย่อมความกดอากาศต่ำกำลังแรง** ที่เคลื่อนตามแนว **ร่องมรสุม** (monsoon trough)
พาดผ่านภาคกลางตอนล่างและภาคตะวันออก ร่วมกับ **มรสุมตะวันตกเฉียงใต้** ที่มีกำลังแรงขึ้น

ผลคือฝนตกหนักต่อเนื่องหลายวันจน กทม. ประกาศพื้นที่ประสบสาธารณภัยครบทั้ง 50 เขต

สื่อเรียกลักษณะนี้ว่า **"ฝนแช่"** ซึ่งต่างจากมหาอุทกภัยปี 2554 ที่เป็นน้ำเหนือหลากลงมา
— ฝนตกลงบนเมืองโดยตรง แล้วระบายไม่ทัน

In [ ]:
print(ec.describe_case("bangkok2026"))

## 4.1 ทำไมเคสนี้ติดตามศูนย์กลางไม่ได้

ลองเรียก `plot_track()` ดู แล้วสังเกตคำเตือนที่ฟังก์ชันพิมพ์ออกมา

หย่อมความกดอากาศต่ำบนร่องมรสุมไม่มีศูนย์กลางชัดเจนแบบพายุหมุนเขตร้อน
จุดต่ำสุดจึงกระโดดไปมา — **เครื่องมือที่ใช้ได้ดีกับเคสหนึ่ง อาจใช้ไม่ได้กับอีกเคส**
การรู้ว่าเครื่องมือพังเมื่อไร สำคัญพอ ๆ กับการรู้วิธีใช้

In [ ]:
ds4 = ec.load_case("bangkok2026")
ax, track4 = ec.plot_track(ds4, "bangkok2026")
plt.show()
print(track4[track4["time"].dt.hour == 0].to_string(index=False))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4.2 ดูที่สนามความกดอากาศและลมแทน

สิ่งที่ควรมองหาคือ **แนวร่องมรสุม** ซึ่งปรากฏเป็นแถบความกดอากาศต่ำพาดตามแนวตะวันออก–ตะวันตก
และ **ลมตะวันตกเฉียงใต้** ที่พัดความชื้นจากทะเลอันดามันและอ่าวไทยเข้ามา

สังเกตว่าเส้นชั้นความกดอากาศในเคสนี้ห่างกันมากเมื่อเทียบกับเคสพายุ
— pressure gradient อ่อนกว่า ลมจึงไม่แรง แต่ฝนกลับหนักกว่า

In [ ]:
d0_bkk = ec.case_info("bangkok2026")["d0"]
d = ds4.sel(time=f"{d0_bkk}T06:00", level=850)

ax = eu.make_map_axes(domain=ec.case_info("bangkok2026")["domain"], figsize=(9, 9))
if ec.has_rh(ds4):
    cf = eu.shade(ax, d["r"], cmap="BrBG", levels=np.arange(30, 101, 5))
else:
    print("ไฟล์นี้ไม่มีความชื้นสัมพัทธ์ จึงแรเงาด้วยความเร็วลมแทน")
    cf = eu.shade(ax, eu.wind_speed(d["u"], d["v"]), cmap="YlGnBu",
                  levels=np.arange(0, 26, 2))
eu.streamlines(ax, d["u"], d["v"], density=1.8)
eu.contour(ax, eu.to_hpa(d["msl"]), levels=np.arange(996, 1017, 2),
           color="black", linewidth=0.8)

for name, (lat, lon) in ec.case_info("bangkok2026")["focus_points"].items():
    ax.plot(lon, lat, "r^", markersize=8, transform=ccrs.PlateCarree(), zorder=6)

eu.finish(ax, title=f"น้ำท่วม กทม. — สนามความชื้น/ลม 850 hPa\n{d0_bkk} 06 UTC",
          mappable=cf,
          cbar_label="ความชื้นสัมพัทธ์ (%)" if ec.has_rh(ds4)
                     else "ความเร็วลม (m s$^{-1}$)")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4.3 ฝนสะสม — ตัวชี้วัดที่สำคัญที่สุดของเคสนี้

น้ำท่วมเมืองเกิดจาก **ฝนสะสมเทียบกับความสามารถในการระบายน้ำ** ไม่ใช่ความเร็วลม

In [ ]:
ec.panel_daily(ds4, "bangkok2026", variable="rain")
plt.show()

In [ ]:
ax, acc4 = ec.plot_accumulated_rain(ds4, "bangkok2026",
                                   start="2026-09-23", end="2026-09-27")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4.4 "ฝนแช่" หน้าตาเป็นอย่างไรในข้อมูล

ฝนที่ตกหนักมากในเวลาสั้น ๆ กับฝนปานกลางที่ตกต่อเนื่องหลายวัน สร้างปัญหาคนละแบบ
กราฟบนคือความแรงรายช่วงเวลา กราฟล่างคือยอดสะสม ดูคู่กันจะแยกสองรูปแบบนี้ออก

In [ ]:
pts4 = ec.point_series(ds4, ec.case_info("bangkok2026")["focus_points"])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for place, grp in pts4.groupby("place"):
    a1.plot(grp["time"], grp["rain_mm"], "o-", markersize=3, label=place)
    a2.plot(grp["time"], grp["rain_mm"].cumsum(), "-", linewidth=2, label=place)

a1.set_ylabel("ฝนราย 6 ชม. (mm)")
a1.legend(fontsize=8)
a1.grid(alpha=0.3)
a1.set_title("น้ำท่วม กทม. — ฝนราย 6 ชั่วโมง และฝนสะสม (เส้นแดง = D0)", loc="left")

a2.set_ylabel("ฝนสะสมตั้งแต่ต้นช่วง (mm)")
a2.set_xlabel("เวลา (UTC)")
a2.grid(alpha=0.3)

for a in (a1, a2):
    a.axvline(pd.Timestamp(d0_bkk), color="crimson", linestyle=":", linewidth=2)
fig.autofmt_xdate()
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4.5 เทียบ ERA5 กับสถานีตรวจวัดจริง

ข่าววันที่ 25 ก.ย. 2569 รายงานฝนสะสม 24 ชั่วโมงสูงสุดที่ ปตร.ประชาร่วมใจ เขตมีนบุรี
**101.5 มม.** และที่ ปตร.คลองสามวา **97.5 มม.**

ลองดูว่า ERA5 ให้ค่าเท่าไรที่จุดเดียวกัน

In [ ]:
print(ec.compare_with_gauge(ds4, "bangkok2026").to_string(index=False))

**ค่ามักไม่ตรงกัน และนั่นถูกต้องแล้ว** เพราะทั้งสองอย่างวัดคนละสิ่ง

| | สถานีตรวจวัด | ERA5 |
|---|---|---|
| วัดอะไร | ฝนที่ตกลงบนจุดเดียว กว้างไม่กี่ตารางเมตร | ฝนเฉลี่ยทั้งกล่องกริด ประมาณ 28 × 28 กม. |
| ได้มาอย่างไร | เครื่องมือวัดโดยตรง | แบบจำลองที่กลืนข้อมูลตรวจวัดเข้าไป |
| จุดอ่อน | เป็นตัวแทนพื้นที่รอบ ๆ ได้แค่ไหนไม่แน่ | เฉลี่ยจนยอดฝนหายไป |

ฝนจากเมฆ convective ในเขตร้อนมักตกเป็นหย่อมกว้างไม่กี่กิโลเมตร
**กริด 0.25 องศาจึงมองไม่เห็นยอดฝนระดับเขต** ค่าที่ได้จะต่ำกว่าสถานีแทบทุกครั้ง

ข้อสรุปสำหรับงานภูมิศาสตร์: ใช้ ERA5 เพื่อเข้าใจ **กลไกระดับใหญ่** (ลม ความชื้น ร่องมรสุม)
แล้วใช้ข้อมูลสถานีหรือเรดาร์เพื่อดู **ความรุนแรงระดับท้องถิ่น** — ใช้แทนกันไม่ได้

> นี่คือเหตุผลที่งานวิจัยด้านฝนหนักมักใช้ข้อมูลหลายแหล่งร่วมกันเสมอ
> และเป็นเหตุผลที่เรดาร์ตรวจอากาศยังจำเป็น แม้จะมี reanalysis ที่ครอบคลุมทั้งโลกแล้ว

## 4.6 อธิบายผลลัพธ์

**คำถามนำ**

1. ฝนสะสมตลอดช่วง 23–27 ก.ย. ที่จุดใน กทม. เป็นเท่าไร เทียบกับฝนทั้งเดือนกันยายน
   ตามค่าปกติ (ประมาณ 330 มม.) แล้วคิดเป็นกี่เปอร์เซ็นต์ของทั้งเดือน
2. ดูกราฟฝนสะสม — ฝนตกเป็นก้อนใหญ่ครั้งเดียว หรือทยอยตกต่อเนื่อง
   รูปแบบไหนที่ระบบระบายน้ำของเมืองรับมือยากกว่า เพราะอะไร
3. ความชื้นที่ 850 hPa เหนือภาคกลางเริ่มสูงขึ้นตั้งแต่ D เท่าไร
   ลมตะวันตกเฉียงใต้แรงขึ้นเมื่อไร ถ้าใช้สองตัวนี้เป็นสัญญาณ จะเตือนล่วงหน้าได้กี่วัน
4. เคสนี้ไม่มีพายุที่มีชื่อ ทำให้การสื่อสารความเสี่ยงต่อสาธารณะยากกว่าสามเคสแรกอย่างไร
5. ถ้าจะทำแผนที่เสี่ยงน้ำท่วมของ กทม. ข้อมูล ERA5 ใช้ตอบอะไรได้บ้าง
   และต้องหาข้อมูลอะไรเพิ่มที่ ERA5 ตอบไม่ได้

### ✍️ ข้อสรุปของฉัน — กรณีน้ำท่วม กทม.

(เขียนที่นี่)

---

---

# เปรียบเทียบทั้งสี่เหตุการณ์

In [ ]:
datasets = {"dianmu2021": ds1, "noru2022": ds2,
            "yagi2024": ds3, "bangkok2026": ds4}
comparison = ec.compare_cases(datasets)
display(comparison)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
labels = [ec.case_info(c)["name_th"] for c in ec.CASE_ORDER]

axes[0].bar(labels, comparison["ความกดต่ำสุด (hPa)"], color="navy")
axes[0].set_title("ความกดอากาศต่ำสุด (hPa)")
axes[0].set_ylim(comparison["ความกดต่ำสุด (hPa)"].min() - 10, 1010)

axes[1].bar(labels, comparison["ฝนรวมสูงสุดในพื้นที่ศึกษา (mm)"], color="teal")
axes[1].set_title("ฝนรวมสูงสุดในพื้นที่ศึกษา (mm)")

axes[2].bar(labels, comparison["ลมแรงสุด 850 hPa (m/s)"], color="darkred")
axes[2].set_title("ลมแรงสุด 850 hPa (m s$^{-1}$)")

for ax in axes:
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ✍️ ข้อสรุปรวม

ตอบสามคำถามนี้โดยอ้างตัวเลขจากตารางเปรียบเทียบ

1. พายุที่ความกดอากาศต่ำที่สุด ทำให้ฝนในไทยมากที่สุดด้วยหรือไม่ ถ้าไม่ อะไรคือคำอธิบาย
2. ถ้าต้องเลือก **ตัวแปรเดียว** ไว้เฝ้าระวังล่วงหน้า 3 วัน จะเลือกตัวไหน เพราะอะไร
3. เคสที่ 4 ไม่มีพายุที่มีชื่อ แต่ความเสียหายสูงมาก — ข้อนี้บอกอะไรเกี่ยวกับ
   การใช้ "ชื่อพายุ" เป็นตัวตั้งในการเฝ้าระวัง
4. ความรู้จากสี่เคสนี้ ใช้กับพื้นที่ที่นักศึกษาสนใจได้อย่างไร

> **ข้อควรระวังทางวิชาการ:** สี่เหตุการณ์คือกลุ่มตัวอย่างที่เล็กมาก
> ใช้สรุปแนวโน้มเชิงสถิติไม่ได้ ใช้ได้เพียงเพื่อ **เข้าใจกลไก** ของแต่ละกรณี
> ถ้าจะสรุปว่า "พายุแบบนี้มักทำให้เกิดอะไร" ต้องวิเคราะห์หลายสิบเหตุการณ์ขึ้นไป

---

## แบบฝึกหัดท้ายบท

1. คำนวณ **ความเร็วการเคลื่อนที่ของพายุ** (km/h) ของสามเคสแรก จากตาราง track
   (เคสที่ 4 ไม่มีศูนย์กลางชัดเจน ให้ข้ามไป)
   โดยใช้ `eu.haversine` หาระยะระหว่างตำแหน่งที่ติดกัน แล้วหารด้วย 6 ชั่วโมง
   พล็อตเป็นกราฟเส้นเทียบกันทั้งสามเคส โดยใช้ `day_offset` เป็นแกน x
2. พล็อต **หน้าตัดแนวดิ่ง** (vertical cross-section) ของความชื้นสัมพัทธ์ตามแนวละติจูด
   ที่ผ่านศูนย์กลางพายุในวัน D0 ของเคสที่ 1
   (ใบ้: `ds1["r"].sel(latitude=..., method="nearest")` จะเหลือมิติ `level × longitude`)
3. หาว่า **ฝนสะสม 3 วัน** (D-1 ถึง D+1) สูงสุดของแต่ละเคสอยู่ที่พิกัดใด
   แล้ววางจุดทั้งสี่ลงบนแผนที่ใบเดียวกัน
4. เลือกจังหวัดหนึ่งที่สนใจ ดึง time series ของฝนและความกดอากาศทั้งสี่เคสมาเทียบกัน
   โดยวาง D0 ให้ตรงกัน แล้วอธิบายว่าเคสไหนกระทบจังหวัดนั้นมากที่สุด

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

---

### เฉลยแบบฝึกหัด

ผู้สอนจะแจกเฉลยหลังหมดเวลาทำแบบฝึกหัดในคาบ

> ลองด้วยตัวเองให้สุดก่อน การติดแล้วแก้ได้เองคือช่วงที่สมองเรียนรู้มากที่สุด
> ถ้าติดเกิน 10 นาทีให้ยกมือถามได้เลย ไม่ต้องรอ


---

## ภาคผนวก ก — สำหรับผู้สอน: เตรียมไฟล์ข้อมูลกรณีศึกษา

ไฟล์ใน `data/cases/` สร้างด้วย `utils/fetch_case_data.py` ซึ่ง **รันครั้งเดียวก็พอ**
ผลลัพธ์ commit ขึ้น GitHub แล้วนักศึกษาโหลดใช้ได้เลย ไม่ต้องสมัคร CDS ไม่ต้องรอคิว

### วิธีที่ 1 — Google Cloud (ไม่ต้องสมัครอะไรเลย)

ARCO-ERA5 เป็นสำเนา ERA5 ที่เปิดให้เข้าถึงสาธารณะบน Google Cloud

```python
!pip install -q gcsfs zarr xarray netcdf4
!wget -q https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/utils/era5_utils.py
!wget -q https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/utils/era5_cases.py
!wget -q https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/utils/fetch_case_data.py
!python fetch_case_data.py --source gcs --outdir data/cases
```

### วิธีที่ 2 — Copernicus CDS (ต้องสมัครบัญชีฟรี)

1. สมัครที่ [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu)
2. copy **API key** จากหน้า profile
3. เข้าหน้า dataset แล้วกด **ยอมรับ license** ที่ท้ายหน้า — **ข้อนี้ลืมกันบ่อยที่สุด**
   ถ้าไม่กด จะดาวน์โหลดไม่ได้และข้อความ error ไม่บอกสาเหตุชัด

```python
!pip install -q cdsapi
with open("/root/.cdsapirc", "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write("key: ใส่ API key ของคุณที่นี่\n")

!python fetch_case_data.py --source cds --outdir data/cases
```

เสร็จแล้วดาวน์โหลดไฟล์ใน `data/cases/` ลงเครื่อง แล้ว commit ขึ้น repo ฝั่งนักศึกษา

### ข้อควรรู้เมื่อใช้ ERA5 เอง

- **เวลาเป็น UTC เสมอ** เวลาไทย = UTC + 7 ชั่วโมง ประกาศกรมอุตุฯ เป็นเวลาไทย
  ต้องลบ 7 ก่อนเทียบกับข้อมูล — ประกาศ 16:00 น. ของไทย คือ 09 UTC
- `area` ของ CDS เรียง **เหนือ ตะวันตก ใต้ ตะวันออก** ไม่ใช่ lon ก่อน lat
- `tp` เป็นฝนสะสมรายชั่วโมงหน่วยเมตร ต้องรวมเองก่อนใช้ — `fetch_case_data.py` ทำให้แล้ว
- ขอเฉพาะโดเมนและตัวแปรที่ใช้จริง ขอทั้งโลกทุกตัวแปรจะได้ไฟล์หลายสิบ GB

---

## ภาคผนวก ข — เพิ่มเหตุการณ์ใหม่เอง

เปิด `utils/era5_cases.py` แล้วเพิ่ม entry ใน `CASES` ตามแบบของเคสที่มีอยู่
ต้องมีอย่างน้อย: `id`, `name_th`, `d0`, `start`, `end`, `domain`, `focus_domain`,
`focus_points`, `timeline`, `sources`

จากนั้นรัน `fetch_case_data.py --cases <id ใหม่>` แล้ว commit ไฟล์ขึ้น repo
โค้ดวิเคราะห์ทั้งหมดในบทนี้ใช้กับเคสใหม่ได้ทันทีโดยไม่ต้องแก้อะไรเลย

ถ้าเคสใหม่ไม่ใช่พายุหมุนเขตร้อน ให้ใส่ `"kind": "monsoon_low"`
ฟังก์ชัน `plot_track()` จะเตือนเองว่าไม่ควรใช้การติดตามศูนย์กลางกับเคสแบบนั้น

เหตุการณ์ที่น่าเพิ่มต่อ: พายุโพดุล (ส.ค. 2562), พายุโนอึล (ก.ย. 2563),
หรือเหตุการณ์ในพื้นที่ที่นักศึกษาสนใจเอง

### เหตุการณ์ที่เพิ่งเกิดไม่นาน

ERA5 รุ่นสมบูรณ์ออกช้ากว่าเวลาจริงราว 2–3 เดือน ส่วนรุ่นเบื้องต้น (ERA5T) ออกเร็วกว่ามาก
และเข้าถึงได้ผ่าน CDS แต่สำเนาบน Google Cloud มักตามหลังหลายเดือน

ดังนั้นเคส `bangkok2026` ให้ใช้ `--source cds` ถ้า `--source gcs` แจ้งว่าข้อมูลมีไม่ถึงวันที่ต้องการ

---

## ไปต่อทางไหนได้อีก

- **MetPy** — คำนวณ CAPE, vorticity, moisture flux convergence จากข้อมูลกริด
- **GeoPandas** — ซ้อนขอบเขตจังหวัด แล้วหาฝนเฉลี่ยรายจังหวัดเพื่อเทียบกับรายงานความเสียหาย
- **IBTrACS** — ฐานข้อมูลเส้นทางพายุอย่างเป็นทางการของ NOAA ใช้ตรวจสอบเส้นทางที่เราคำนวณเอง
- **xarray + dask** — จัดการข้อมูลหลายสิบปี เพื่อหาว่าพายุแบบนี้เกิดบ่อยแค่ไหน